In [ ]:
using Pkg
using CSV
using DataFrames
using Plots

In [ ]:
function read_mfem_nodes(filename)
    coords = Float64[]
    in_nodes = false
    skip = 0

    for line in eachline(filename)
        if strip(line) == "nodes"
            in_nodes = true
            skip = 4  # skip FE space metadata
            continue
        end

        if in_nodes
            if skip > 0
                skip -= 1
                continue
            end
            try
                push!(coords, parse(Float64, strip(line)))
            catch
            end
        end
    end
    return coords
end

In [ ]:
function read_other(filename)
    coords = Float64[]
    in_nodes = false
    skip = 4

    for line in eachline(filename)
        if skip > 0
            skip -= 1
            continue
        end
        try
            push!(coords, parse(Float64, strip(line)))
        catch
        end
    end
    return coords
end

In [ ]:
# Read Brook
data = CSV.read("snapshot_smooth_shu_osher_p1/snapshot_smooth_shu_osher_p1/t_0-1/num_smooth_shu_osher_rk3_1d_m_4096.csv", DataFrame)  # if using CSV.jl
xB = data[:,1]; rhoB = data[:,2]; muB = data[:,3]; EB = data[:,4];
vB = data[:,5]; pB = data[:,6]; SigmaB = data[:,7]; eB = data[:,8];

In [ ]:
# Read Dave
x = read_mfem_nodes("Laghos_1_mesh"); rho = read_other("Laghos_1_rho")
v = read_other("Laghos_1_v"); e = read_other("Laghos_1_e")
igrp = read_other("Laghos_1_igr");

In [ ]:
pv = sortperm(x)
x = x[pv]; rho = rho[pv]; v = v[pv]; e = e[pv]
p = 0.4 .* rho .* e
igrp = igrp[pv];

In [ ]:
x4 = copy(x); p4 = copy(p); e4 = copy(e); v4 = copy(v); rho4 = copy(rho);

In [ ]:
plot(x3, v3, xlim = (0.37,0.67), label = "SSP-RK3 + (3,3) Order", title= "Shu Osher t = 0.1", ylab = "Velocity", ylim = (2.0,2.8))
plot!(x1, v1, label = "SSP-RK3 + (2,1) Order")
plot!(xB, vB, label = "Brook")

In [ ]:
plot(x1, rho1, label = "SSP-RK3", title= "Shu Osher t = 0.1", ylab = "Density", xlim = (0.37,0.67))
plot!(x2, rho2, label = "RK4")
plot!(xB, rhoB, label = "Brook")

In [ ]:
p1 = plot(x, rho, title = "Density", label = "")
plot!(xB, rhoB)
p2 = plot(x, v, title = "Velocity", label = "")
plot!(xB, vB)
p3 = plot(x, e, title = "Internal Energy", label = "")
plot!(xB, eB)
p4 = plot(x, p, title = "Pressure", label = "")
plot!(xB, pB)

plot(p1, p2, p3, p4, layout = (2, 2))

In [ ]:
p1 = plot(x1, rho1, title = "Density", label = "")
p2 = plot(x1, v1, title = "Velocity", label = "")
p3 = plot(x1, e1, title = "Internal Energy", label = "")
p4 = plot(x1, p1, title = "Pressure", label = "")

plot(p1, p2, p3, p4, layout = (2, 2))

In [ ]:
# Can also compare to exact Riemann solver sometimes from Julia
include("RiemannSolver.jl")

In [ ]:
#                   rho,   u,  p, gamma
left  = HydroStatus(1.1, 0.0, 0.11, 1.4)
right = HydroStatus(0.1, 0.0, 0.01, 1.4)
xs = range(0.0, 1.0, length=2052)
t  = 0.2

states = sample_riemann(xs .- 0.5, t, left, right);

In [ ]:
ExRho = getfield.(states, :rho)
ExU = getfield.(states, :u)
ExP = getfield.(states, :p)
ExE = ExP ./ (0.4 .* ExRho);

In [ ]:
p1 = plot(xs, ExRho, title = "Density")
p2 = plot(xs, ExU, title = "Velocity")
p3 = plot(xs, ExE, title = "Internal Energy")
p4 = plot(xs, ExP, title = "Pressure")

plot(p1, p2, p3, p4, layout = (2, 2))

In [ ]:
data = CSV.read("snapshot_smooth_shu_osher_p1/snapshot_smooth_shu_osher_p1/t_0-2/num_smooth_shu_osher_rk3_1d_m_4096.csv", DataFrame)  # if using CSV.jl
xB = data[:,1]; rhoB = data[:,2]; muB = data[:,3]; EB = data[:,4];
vB = data[:,5]; pB = data[:,6]; SigmaB = data[:,7]; eB = data[:,8];

In [ ]:
#plot(xB, yB, rhoB, st =:surface, camera=(0, 90))

In [ ]:
p1 = plot(x, rho, title = "Density", label = "")
plot!(xB, rhoB)
p2 = plot(x, v, title = "Velocity", label = "")
plot!(xB, vB)
p3 = plot(x, e, title = "Internal Energy", label = "")
plot!(xB, eB)
p4 = plot(x, p, title = "Pressure", label = "")
plot!(xB, pB)

plot(p1, p2, p3, p4, layout = (2, 2))

In [ ]:
#Interpolate Functions

ConvInt(a, b, c) = (b-a)/(c-a)
ConvAvg(a, b, L) = (1-L)*a + L*b 

point = 1
stay = true
igrp2 = []
for i in 1:length(xB)-1
    while(stay)
        if(x[point + 1] > xB[i])
            L = ConvInt(x[point], xB[i], x[point + 1])
            push!(igrp2, ConvAvg(igrp[point], igrp[point + 1], L))
            stay = false
        else
            point += 1
        end
    end
    stay = true
end